# Step 8 — track gauge per stop

Tags each catalog stop with the gauge(s) of the tracks serving it, from OSM's
`gauge` tag on `railway=rail|light_rail` ways near the stop. A stop can carry
**several** gauges — Iberian stations often have 1435 alongside 1668, the
Baltics 1520 alongside 1435 — so the column is a set, not a value.

Only gauges a night train can physically run on are kept: `railway=rail`
tracks with **gauge ≥ 1435 mm**. Anything narrower is trams, Stadtbahn or
narrow-gauge regional railways sharing the station forecourt (Bielefeld's
Stadtbahn, Alicante's TRAM, the metre-gauge RhB at Chur) — real tracks, but
not infrastructure the target network's rolling stock will ever touch. Both
filters are needed: light-rail is excluded by tag, but narrow-gauge mainlines
are tagged `railway=rail` too, so the millimetre threshold does the rest.

**Why:** routing on OpenRailRouting fails on non-1435 networks unless the
composition's gauge capability matches the track, and the gauge filter in
`custom_models/night_train.json` needs to become composition-level. Both need
to know, per stop, what gauge the infrastructure actually offers.

**Input:** `data/step7_enriched_stops.csv` (the catalog stops).
**Output:** `data/step8_stop_gauges.csv` — `stop_id, gauges, tagged_tracks,
untagged_tracks, gauge_source`.

The station extract only contains station objects, not track ways, so the
tracks come from Overpass — batched, resumable, and polite, following
step 3a's conventions. A full run is ~1,000 stops in ~25 batches. Re-running
the fetch cell only fills in stops the output does not have yet; delete the
output file for a full refresh.

`gauge_source` is honest about evidence: `tagged` (at least one nearby track
carries the tag), `untagged_tracks` (rail is there, gauge is not mapped — the
consumer decides what to assume), `no_tracks_nearby` (probably a coordinate
sitting on the building rather than the platforms — worth a look).
Cross-validating against the OpenRailRouting graph is the planned second
signal and stays an open item in the README.


## Imports and parameters

In [1]:
import csv
import json
import time
import urllib.request
from collections import Counter

from data_sources import DATA_DIR, local_input

TRACK_RADIUS_M = 150  # platforms sit within this of the station object
MIN_GAUGE_MM = 1435  # night-train rolling stock: standard and broad gauge only
BATCH_SIZE = 40  # stops per Overpass request
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

INPUT_PATH = local_input("step7_enriched_stops.csv", "step7_enrich_stops.ipynb")
OUTPUT_PATH = DATA_DIR / "step8_stop_gauges.csv"
FIELDNAMES = ["stop_id", "gauges", "tagged_tracks", "untagged_tracks", "gauge_source"]

# 2026-08-29: four of the no_tracks_nearby stops corrected. In every case
# the station node itself is right (all classify heavy_rail in step 3b,
# nothing better sits nearby) — OSM simply carries no gauge-tagged rail
# way within TRACK_RADIUS_M. PL/TR are plain 1435 networks; Росинка sits
# on Ukraine's uniformly 1520 network with sparse tagging. Three more of
# the original ten (Oldenburg (Holstein), Краматорськ, Слов'янськ) left
# the catalog with the schedule and need no override. NOT overridden,
# deliberately: GR Ρίο (osm:n9721698903 — metre-gauge line, no >=1435
# track exists) and AL Durrës (osm:n13895194677 — network out of
# service, tagged disused); their NULLs are true statements.
# Overrides apply at fetch time — adding one for an already-done stop requires
# deleting its row from the output CSV first.
# RESUME RULE: overrides apply at fetch time only — adding one for a stop
# already in the output CSV requires deleting that stop's row there first,
# or the old row survives untouched.
GAUGE_OVERRIDES: dict[str, str] = {
    "osm:n563273123": "1435",  # Łeba, PL
    "osm:n2726063373": "1435",  # Adana, TR
    "osm:n2596542075": "1435",  # Mersin Garı, TR
    "osm:n8220188327": "1520",  # Росинка, UA
    "osm:w80588808": "1520",  # Миколаїв, UA — city of half a million; the
    # station object is a way, no gauge-tagged track within the radius
    "osm:w600545583": "1435",  # Дулово, BG — tracks present but untagged;
    # the Bulgarian network is uniformly 1435
    "osm:n1132544592": "1435",  # Садово, BG — Plovdiv–Burgas main line
    "osm:n2235196597": "1435",  # Dej Călători, RO — added in step 6 after
    # the bus-station mismatch; Romanian network is uniformly 1435
}

## Stops to process

The stop list comes from step 7's output, and the coordinates from the
qualification layers it was built from — so gauge rows always describe the
same objects the catalog seeds.


In [2]:
stops = []
with open(
    local_input("step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    coords = {
        row["osm_stop_id"]: (float(row["osm_lat"]), float(row["osm_lon"]))
        for row in csv.DictReader(fh)
    }
with open(
    local_input("step6_manual_additions.csv", "step6_manual_additions.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        coords.setdefault(
            row["stop_id"], (float(row["stop_lat"]), float(row["stop_lon"]))
        )

with open(INPUT_PATH, encoding="utf-8-sig", newline="") as fh:
    for row in csv.DictReader(fh):
        lat, lon = coords[row["stop_id"]]
        stops.append(
            {
                "stop_id": row["stop_id"],
                "name": row["stop_name"],
                "lat": lat,
                "lon": lon,
            }
        )

done = set()
if OUTPUT_PATH.is_file():
    with open(OUTPUT_PATH, encoding="utf-8-sig", newline="") as fh:
        done = {row["stop_id"] for row in csv.DictReader(fh)}
unknown_gauge_overrides = sorted(set(GAUGE_OVERRIDES) - {s["stop_id"] for s in stops})
if unknown_gauge_overrides:
    raise KeyError(f"GAUGE_OVERRIDES for unknown stops: {unknown_gauge_overrides}")

todo = [s for s in stops if s["stop_id"] not in done]
print(f"{len(stops)} stops, {len(done)} already done, {len(todo)} to fetch")

1179 stops, 1053 already done, 126 to fetch


## Fetch tracks per batch

One request covers `BATCH_SIZE` stops: a union of `way(around:...)` clauses.
Results come back with geometry centers so each way is attributed to the
nearest stop in the batch — good enough at 150 m, and it avoids one request
per stop.


In [3]:
def normalise_gauge(value: str) -> tuple[list[str], int]:
    """OSM gauge values: numbers, semicolon lists, or words. Words map to the
    number where unambiguous; non-numeric leftovers are kept verbatim rather
    than guessed, so a weird value is visible downstream instead of laundered.
    Returns (kept gauges, count of narrow values dropped by MIN_GAUGE_MM)."""
    kept, narrow = [], 0
    for part in str(value).split(";"):
        part = part.strip()
        if part == "standard":
            part = "1435"
        if not part:
            continue
        try:
            if int(part) < MIN_GAUGE_MM:
                narrow += 1
                continue
        except ValueError:
            pass  # non-numeric ("broad"): keep visible
        kept.append(part)
    return kept, narrow


# railway=rail only: light_rail brought metre-gauge Stadtbahn/tram-train
# tracks running past mainline stations into the gauge sets (Bielefeld
# 1000;1435, Alacant 1000;...) — urban rail is not what a night train can use.
def fetch_batch(batch):
    clauses = "".join(
        f'way(around:{TRACK_RADIUS_M},{s["lat"]:.7f},{s["lon"]:.7f})["railway"="rail"];'
        for s in batch
    )
    query = f"[out:json][timeout:600];({clauses});out tags center;"
    req = urllib.request.Request(
        OVERPASS_URL,
        data=query.encode(),
        headers={"User-Agent": "night-train-target-network stop pipeline"},
    )
    with urllib.request.urlopen(req, timeout=600) as resp:
        return json.load(resp).get("elements", [])


def nearest_stop(batch, lat, lon):
    return min(batch, key=lambda s: (s["lat"] - lat) ** 2 + (s["lon"] - lon) ** 2)


write_header = not OUTPUT_PATH.is_file()
failed_batches = 0
with open(OUTPUT_PATH, "a", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
    if write_header:
        writer.writeheader()
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start : start + BATCH_SIZE]
        for attempt in (1, 2, 3):
            try:
                elements = fetch_batch(batch)
                break
            except Exception as exc:
                print(f"  batch at {start}: attempt {attempt} failed ({exc})")
                time.sleep(20 * attempt)
        else:
            failed_batches += 1
            continue  # resumable: missing stops are re-fetched on the next run

        gauges = {s["stop_id"]: Counter() for s in batch}
        untagged = Counter()
        narrow_only = Counter()
        for el in elements:
            center = el.get("center") or {}
            if "lat" not in center:
                continue
            stop = nearest_stop(batch, center["lat"], center["lon"])
            value = el.get("tags", {}).get("gauge")
            if value:
                kept, narrow = normalise_gauge(value)
                for gauge in kept:
                    gauges[stop["stop_id"]][gauge] += 1
                narrow_only[stop["stop_id"]] += narrow
            else:
                untagged[stop["stop_id"]] += 1

        for s in batch:
            tagged = gauges[s["stop_id"]]
            if s["stop_id"] in GAUGE_OVERRIDES:
                tagged = Counter(GAUGE_OVERRIDES[s["stop_id"]].split(";"))
                source = "override"
            elif tagged:
                source = "tagged"
            elif untagged[s["stop_id"]]:
                source = "untagged_tracks"
            else:
                source = "no_tracks_nearby"
            writer.writerow(
                {
                    "stop_id": s["stop_id"],
                    "gauges": ";".join(sorted(tagged)),
                    "tagged_tracks": sum(tagged.values()),
                    "untagged_tracks": untagged[s["stop_id"]],
                    "gauge_source": source,
                }
            )
        fh.flush()
        print(f"  {min(start + BATCH_SIZE, len(todo))}/{len(todo)}")
        time.sleep(5)

if failed_batches:
    print(f"{failed_batches} batch(es) failed — re-run this cell to fill the gaps")

  batch at 0: attempt 1 failed (HTTP Error 504: Gateway Timeout)
  batch at 0: attempt 2 failed (HTTP Error 504: Gateway Timeout)
  40/126
  batch at 40: attempt 1 failed (HTTP Error 504: Gateway Timeout)
  80/126
  120/126
  batch at 120: attempt 1 failed (HTTP Error 429: Too Many Requests)
  126/126


## Review

The distribution is its own sanity check: overwhelmingly single-gauge 1435,
1668 across Iberia, 1520/1524 in the Baltics and Finland, and a short
multi-gauge list that should read like a who's-who of break-of-gauge stations
(Irun/Hendaye, Cerbère/Portbou, the Ukrainian border points...). Anything
surprising in the multi-gauge list is worth checking in OSM before anyone
routes on it.


In [4]:
rows = []
with open(OUTPUT_PATH, encoding="utf-8-sig", newline="") as fh:
    rows = list(csv.DictReader(fh))
print(f"{len(rows)} stops with gauge rows")
print("by source:", Counter(r["gauge_source"] for r in rows))
narrow_flagged = [r for r in rows if r["gauge_source"] == "narrow_gauge_only"]
if narrow_flagged:
    print(f"REVIEW: {len(narrow_flagged)} stop(s) with only narrow-gauge tracks nearby")
print("by gauge set:", Counter(r["gauges"] for r in rows).most_common(12))

names = {s["stop_id"]: s["name"] for s in stops}
multi = [r for r in rows if ";" in r["gauges"]]
print(f"\n{len(multi)} multi-gauge stops:")
for r in multi:
    print(f"  {names.get(r['stop_id'], '?')[:40]:42} {r['gauges']:18} {r['stop_id']}")

1179 stops with gauge rows
by source: Counter({'tagged': 1163, 'no_tracks_nearby': 8, 'override': 8})
by gauge set: [('1435', 981), ('1520', 80), ('1668', 40), ('1435;1668', 26), ('1524', 22), ('1435;1520', 13), ('', 8), ('1600', 7), ('1520;1524', 2)]

41 multi-gauge stops:
  Narva                                      1520;1524          osm:n529932898
  Tartu                                      1520;1524          osm:n8761131036
  Alacant Terminal                           1435;1668          osm:n10914769161
  Albacete Los Llanos                        1435;1668          osm:n11016395831
  Castelló                                   1435;1668          osm:n13894649638
  Ciudad Real                                1435;1668          osm:n13717016710
  Córdoba Julio Anguita                      1435;1668          osm:n7567516121
  Granada                                    1435;1668          osm:n7201979115
  Huesca                                     1435;1668          osm:n8051540821
  